# Criando um modelo Pydantic

Para definir um modelo Pydantic, herdamos da classe `BaseModel`.

In [1]:
from pydantic import BaseModel

Pense num modelo Pydantic como uma estrutura de dados com campos nomeados.

Além disso, informamos o tipo esperado de cada campo usando *type hints*.

In [2]:
class Person(BaseModel):
    first_name: str
    last_name: str
    age: int

Este é um modelo Pydantic básico. Ele é um modelo Pydantic porque **herda** de `BaseModel`, mas também é uma classe Python comum —
você pode adicionar seus próprios métodos, propriedades etc., se quiser.

Ele tem três campos, e cada campo recebeu um *type hint* indicando o tipo esperado.

Como o Pydantic aproveita o sistema de *type hints* do Python, dá para usar qualquer *type hint* aqui, inclusive tipos customizados,
genéricos etc.

Com o modelo definido, podemos criar instâncias. Existem três formas básicas de fazer isso; a primeira é chamar a classe com argumentos nomeados.

In [3]:
p = Person(first_name="Evariste", last_name="Galois", age=20)

O Pydantic fornece automaticamente as representações `str` e `repr`:

In [4]:
print(f"str(p): {str(p)!r}")

str(p): "first_name='Evariste' last_name='Galois' age=20"


In [5]:
print(f"repr(p): {repr(p)!r}")

repr(p): "Person(first_name='Evariste', last_name='Galois', age=20)"


Podemos inspecionar os campos de um modelo Pydantic com a propriedade `model_fields`.

> 📝 **Nota de revisão (Pydantic 2.11+):** o original acessava `p.model_fields` pela **instância**. Isso está depreciado desde a 2.11
> (será removido na v3) e emite `DeprecationWarning`. O correto é acessar pela **classe**: `Person.model_fields`.

In [6]:
print(f"Person.model_fields: {Person.model_fields!r}")

Person.model_fields: {'first_name': FieldInfo(annotation=str, required=True), 'last_name': FieldInfo(annotation=str, required=True), 'age': FieldInfo(annotation=int, required=True)}


Repare que cada campo aparece listado e, um detalhe interessante, com `required=True`.

E, de fato, se tentarmos criar uma instância sem passar valor para algum desses campos, recebemos uma exceção.

O Pydantic levanta a exceção `ValidationError` sempre que algo dá errado ao criar uma instância do modelo.

In [7]:
from pydantic import ValidationError

Vejamos o que acontece quando não informamos um campo obrigatório:

In [8]:
try:
    Person(last_name='Galois')
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Person
first_name
  Field required [type=missing, input_value={'last_name': 'Galois'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
age
  Field required [type=missing, input_value={'last_name': 'Galois'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


Como você pode ver, o Pydantic reporta **todos** os erros de validação que encontra.

Por padrão, o Pydantic não para de validar no primeiro problema. Ele continua validando tudo e devolve a lista completa de erros.

Isso é muito útil em APIs REST, onde normalmente queremos informar ao cliente **tudo** o que estava errado no payload JSON.

Eu disse que classes Pydantic são classes Python comuns. Elas ganham a funcionalidade especial herdando de `BaseModel`, mas continuam
sendo classes normais — portanto podemos adicionar propriedades e métodos.

In [9]:
class Person(BaseModel):
    first_name: str
    last_name: str
    age: int

    @property
    def display_name(self):
        return f"{self.first_name} {self.last_name[0]}"

In [10]:
p = Person(first_name="Evariste", last_name="Galois", age=20)
print(f"p: {p!r}")

p: Person(first_name='Evariste', last_name='Galois', age=20)


In [11]:
print(f"p.display_name: {p.display_name!r}")

p.display_name: 'Evariste G'


Como em qualquer objeto, também acessamos os campos com notação de ponto:

In [12]:
print(f"p.last_name: {p.last_name!r}")

p.last_name: 'Galois'


E, por padrão, também podemos alterar os valores dos campos:

In [13]:
p.age = 21

In [14]:
print(f"p: {p!r}")

p: Person(first_name='Evariste', last_name='Galois', age=21)


Aqui é preciso um pouco de cuidado. O Pydantic faz validação quando carrega dados (**desserializa**) para criar uma instância.

Em particular, temos *type hints* em cada campo:

In [15]:
try:
    Person(first_name='Evariste', last_name='Galois', age='twenty')
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


Como você vê, a entrada incorreta para `age` foi barrada pelo Pydantic.

Porém, por padrão, isso **não** acontece quando alteramos um campo depois de a instância ter sido criada:

In [16]:
print(f"p: {p!r}")

p: Person(first_name='Evariste', last_name='Galois', age=21)


In [17]:
p.age = "twenty"

In [18]:
print(f"p: {p!r}")

p: Person(first_name='Evariste', last_name='Galois', age='twenty')


Viu? A alteração passou.

Mais adiante no curso veremos como configurar o modelo para que isso também levante `ValidationError` (`validate_assignment`).